# L5: Open Generative UI

In this lesson, you'll move to the far end of the generative UI spectrum. Instead of registered components or schemas, the agent routes to full applications — like Excalidraw — via MCP Apps, the same app protocol used by Claude, ChatGPT, and other AI hosts.

## 📋 Learning Objectives

1. **Enable open generative UI** - Let the agent generate arbitrary UI on the fly
2. **Understand MCP Apps** - The protocol that lets MCP servers deliver interactive apps to AI hosts
3. **Connect an MCP App** - Wire Excalidraw into your CopilotKit application

In [ ]:
# clear your state if running notebook more than once
from helper import reset_lesson
reset_lesson(5)

### Setup Dependencies

In [ ]:
# !pip install -r requirements.txt?

In [ ]:
from helper import install_frontend
install_frontend()

In [ ]:
from helper import load_api_keys
load_api_keys()

## What you'll build

You'll connect an MCP App (Excalidraw) so the agent can launch a full whiteboard inside the chat:

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Show me a simple network diagram of three routers, two laptops and a server using excalidraw</div>

<img src="./images/mcp-apps-whiteboard.png" alt="MCP Apps Whiteboard" style="max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

Then, you'll enable `openGenerativeUI` — letting the agent generate arbitrary UI on the fly:

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Make it rain tacos!</div>

<img src="./images/open-gen-ui.png" alt="Open Generative UI Calculator" style="max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

## What is Open-Ended Generative UI?

Open-ended generative UI is the most flexible pattern in the stack: the agent is not limited to a small set of pre-registered components or a declarative schema, either fixed or dynamically generated.

That flexibility comes from MCP Apps. The agent discovers app tools from an MCP server and can open those apps when they fit the user's request.

Your frontend acts as a host. It does not need to hand-author every possible UI surface ahead of time; instead, it connects the chat runtime to compatible external apps.

## Why use Open-Ended UI?

Open-ended UI is useful when the long tail of user requests is too broad to cover with a fixed component library.

**Pros**
- Extremely flexible: the agent can route users into richer app experiences, not just inline widgets.
- Lower frontend coupling: your host app can gain new capabilities by connecting to MCP servers.
- Good fit for workflows like whiteboarding, design, planning, and other tool-shaped tasks.

**Cons**
- Less control over the final UI than controlled or declarative approaches.
- Quality depends on the connected MCP apps and how well the agent selects them.
- Requires stronger trust, permissions, and integration guardrails.

## Start the backend and frontend 

In [ ]:
# load back end requirements
from backend.server import start_backend
start_backend(port=8005)

In [ ]:
# start the front end
from helper import start_frontend
start_frontend(port=3005)

Finally, you can open a preview to see where things are starting:

In [ ]:
from helper import display_app
display_app(port=3005)

## The MCP Apps Specification

[MCP Apps](https://modelcontextprotocol.io/extensions/apps/overview) are an extension to the Model Context Protocol that lets MCP servers deliver interactive UI to supported hosts.

The architecture has three parts:
- **Server**: exposes tools and UI resources.
- **Host**: embeds the UI in a sandboxed iframe and proxies communication.
- **View**: the app running inside the iframe.

A key design point is progressive enhancement: if the host supports MCP Apps, the tool renders rich UI; if not, it still works as a normal MCP tool with text output.

CopilotKit handles the hosting — you just point it at an MCP server URL.

### Example MCP Apps

There are many examples of MCP Apps in the wild. To get a sense of what's possible, check out [the official repository of MCP App examples](https://github.com/modelcontextprotocol/ext-apps/tree/main/examples).

For this lesson, you'll set up the [Excalidraw MCP App](https://github.com/excalidraw/excalidraw-mcp) for your agent.

## Add an MCP App Server to your app

CopilotKit supports MCP Apps out of the box — just pass the server URL to `CopilotRuntime`:

In [ ]:
%%writefile frontend/server.ts

import { serve } from "@hono/node-server";
import {
  CopilotRuntime,
  createCopilotEndpoint,
} from "@copilotkit/runtime/v2";
import { LangGraphHttpAgent } from "@copilotkit/runtime/langgraph";

const appAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8005",
});

const runtime = new CopilotRuntime({
  agents: { default: appAgent },
  mcpApps: {
    servers: [
      {
        type: "http",
        url: "https://mcp.excalidraw.com", // <- Exalidraw MCP Server
        serverId: "example_mcp_server",
      },
    ],
  },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4005 }, () => {
  console.log("✓ CopilotKit API server running at http://localhost:4005");
});

A few things to note about the `mcpApps` configuration:

- `mcpApps.servers` connects the runtime to one or more MCP servers that expose app tools.
- Each entry tells the runtime where to discover compatible MCP Apps. Here, you connect to Excalidraw over HTTP.
- CopilotKit automatically augments the agent with MCP app discovery so it can surface those tools during a run.

## Give it a try!

Your agent can now launch Excalidraw inside the chat. Try it:

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Show me a simple network diagram of three routers, two laptops and a server using excalidraw</div>

In [ ]:
from helper import display_app
display_app(port=3005)

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Add labels, titles and please make it more coherent</div>

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-left: 4px solid #f0a020; border-radius: 12px; padding: 16px 20px; margin: 12px 0; font-size: 14px; color: #333; line-height: 1.6;">

**⚠️ Heads up: Open Generative UI is unpredictable**

Because the agent generates the UI from scratch on every request, results may not match what you expect on the first try. You may need to iterate on the prompt a few times before the diagram looks the way you want.

This is one of the core trade-offs of fully open Generative UI: you get maximum flexibility, but you sacrifice consistency and predictability. For UIs that need to be reliable every time, controlled or declarative approaches are the better fit.

</div>

### Enable `openGenerativeUI`

With `openGenerativeUI: true`, the agent can generate arbitrary UI — HTML, CSS, JavaScript — and render it directly in the chat. One line to enable:

In [ ]:
%%writefile frontend/server.ts

import { serve } from "@hono/node-server";
import {
  CopilotRuntime,
  createCopilotEndpoint,
} from "@copilotkit/runtime/v2";
import { LangGraphHttpAgent } from "@copilotkit/runtime/langgraph";

const appAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8005",
});

const runtime = new CopilotRuntime({
  agents: { default: appAgent },
  openGenerativeUI: true,
  mcpApps: {
    servers: [
      {
        type: "http",
        url: "https://mcp.excalidraw.com", // <- Exalidraw MCP Server
        serverId: "example_mcp_server",
      },
    ],
  },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4005 }, () => {
  console.log("✓ CopilotKit API server running at http://localhost:4005");
});

## Give it a try!

Your agent can now generate arbitrary UI on the fly. Try something creative:

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Make it rain tacos!</div>

In [ ]:
from helper import display_app
display_app(port=3005)

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Make a card with an animation of raining taco emojis</div>

## What you learned

- Open generative UI removes the constraint of pre-registered components and schemas — the agent can launch full apps or generate arbitrary UI.
- MCP Apps let an agent surface rich app experiences (like Excalidraw) through a standard protocol.
- `CopilotRuntime` handles MCP app discovery — you just pass the server URL.
- `openGenerativeUI: true` enables the agent to generate and render HTML/CSS/JS directly in the chat.

## Next step

In **Lesson 6**, you'll move beyond generative UI and build a fullstack todo app with **shared state** and **frontend tools**. The agent and the UI share the same live state — both sides can read and write, and CopilotKit keeps them in sync.